In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from typing import List, Optional

base_path = "."

print(base_path)
sys.path.append(base_path)

In [ ]:
!pwd

In [ ]:
# !pip install imblearn
# !pip install gensim
!pip install simpletransformers

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

pd.set_option("display.max_columns", None)
import pandas as pd

from src.preprocessing.consts import Datasets
from src.preprocessing.read_raw_data import read_weak_labeled_csv

df = read_weak_labeled_csv(Datasets.NELA_ELECTIONS_2020.weak_labeled_path, train_df_name=Datasets.NELA_COVID_2021.name)
df = df[["title", "content", "actual_label", "snorkel_majority_vote_label"]].copy()
df = df.drop(df[df.snorkel_majority_vote_label == -1].index)
df = df.rename(columns={"snorkel_majority_vote_label": 'weak_label', "content": "text"})
print(len(df))

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

under_sampler = RandomUnderSampler(random_state=42)
undersampled_df, undersampled_labels = under_sampler.fit_resample(df, df["weak_label"])

In [ ]:
print(undersampled_df["weak_label"].value_counts())
undersampled_df.head()

In [ ]:
undersampled_df["text"] = undersampled_df["title"] + ". " + undersampled_df["text"]

In [ ]:
undersampled_df.drop(["title"], axis=1, inplace=True)

In [ ]:
undersampled_df.head()

# Add Label Column and create concat dataframe

In [ ]:
train_df, test_df = train_test_split(undersampled_df, test_size=0.2, stratify=undersampled_df['weak_label'], random_state=42)

In [ ]:
train_df = train_df.rename(columns={"weak_label": 'label'})
train_df.drop(["actual_label"], axis=1, inplace=True)
train_df.head()

In [ ]:
test_df_with_actual_labels = test_df.copy()
test_df_with_actual_labels.rename(columns={"actual_label": 'label'}, inplace=True)
test_df_with_actual_labels.drop(["weak_label"], axis=1, inplace=True)
test_df_with_actual_labels.head()

In [ ]:
test_df_with_weak_labels = test_df.copy()
test_df_with_weak_labels.rename(columns={"weak_label": 'label'}, inplace=True)
test_df_with_weak_labels.drop(["actual_label"], axis=1, inplace=True)
test_df_with_weak_labels.head()

In [ ]:
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from simpletransformers.classification import ClassificationModel, ClassificationArgs

In [ ]:
import torch

is_cuda = torch.cuda.is_available()
if is_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    
print(device)

In [ ]:
model_args = ClassificationArgs(num_train_epochs=1, overwrite_output_dir=True, manual_seed=42)
model = ClassificationModel(model_type='roberta', model_name='roberta-base', use_cuda=False, num_labels=2, args=model_args)

In [ ]:
model.train_model(train_df)